<table dir="ltr" width="100%">
<thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead>
<tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h1>Day 4 · Streaming and quality reference</h1><p>Executed source reference only. No Kafka, Spark, Delta or Great Expectations execution is claimed.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h1>اليوم الرابع · مرجع التدفق والجودة</h1><p>مرجع مصدر منفذ فقط؛ لا يُدّعى تشغيل Kafka أوSpark أوDelta أوGreat Expectations.</p></td></tr></tbody>
</table>

<table dir="ltr" width="100%">
<thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead>
<tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><p><a href="README.md">Day 4</a> · <a href="SOURCES.md">Sources</a></p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><p><a href="README.md">اليوم الرابع</a> · <a href="SOURCES.md">المصادر</a></p></td></tr></tbody>
</table>



<table dir="ltr" width="100%">
<thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead>
<tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>1. Setup and source integrity</h2></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>1. الإعداد وسلامة المصدر</h2></td></tr></tbody>
</table>



In [1]:
from pathlib import Path
import sys, json
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "course.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the complete course repository")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
SOURCE = ROOT / "data/masar-small-v1"
from masar.workspace import require_fixed_dataset
require_fixed_dataset(SOURCE)
print("MASAR_SMALL_V1: original source hashes verified")

MASAR_SMALL_V1: original source hashes verified


<table dir="ltr" width="100%">
<thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead>
<tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>2. Derive the Day 4 reference</h2></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>2. اشتقاق مرجع اليوم الرابع</h2></td></tr></tbody>
</table>



In [2]:
from masar.trust_reference import day04_reference
reference = day04_reference(SOURCE)
print(json.dumps({k: reference[k] for k in ["scope", "engine_executed", "kafka_executed", "gx_executed", "deliveries", "distinct_events"]}, indent=2))

{
  "scope": "DAY04_SOURCE_EXPECTATIONS_ONLY",
  "engine_executed": false,
  "kafka_executed": false,
  "gx_executed": false,
  "deliveries": {
    "base": 216,
    "replay": 2,
    "late": 1,
    "total": 219
  },
  "distinct_events": 217
}


<table dir="ltr" width="100%">
<thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead>
<tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>3. Explain messages versus events</h2></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>3. تفسير الرسائل مقابل الأحداث</h2></td></tr></tbody>
</table>



In [3]:
from masar.trust_reference import events_from_source, unique_events, GPS_FILES
all_events = []
for filename in GPS_FILES:
    all_events.extend(events_from_source(SOURCE, filename))
    print(f"{filename}: source deliveries={len(all_events)}, distinct event IDs={len(unique_events(all_events))}")
print("These are source-derived counts, not observed Kafka offsets or checkpoint recovery.")

gps.ndjson: source deliveries=216, distinct event IDs=216
gps_replay.ndjson: source deliveries=218, distinct event IDs=216
gps_late.ndjson: source deliveries=219, distinct event IDs=217
These are source-derived counts, not observed Kafka offsets or checkpoint recovery.


<table dir="ltr" width="100%">
<thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead>
<tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>4. Apply the explicit quality policy</h2></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>4. تطبيق سياسة الجودة المعلنة</h2></td></tr></tbody>
</table>



In [4]:
from masar.delta_reference import day03_reference
from masar.silver_reference import read_csv, drivers_index
from masar.trust_reference import quality_probe_rows, evaluate_quality
trusted = day03_reference(SOURCE)["expected_corrected_rows"]
driver_ids = set(drivers_index(read_csv(SOURCE, "drivers.csv")))
clean = evaluate_quality(trusted, driver_ids)
mixed = evaluate_quality(trusted + quality_probe_rows(SOURCE), driver_ids)
rechecked = evaluate_quality(mixed["accepted"], driver_ids)
for label, result in [("trusted", clean), ("mixed", mixed), ("rechecked", rechecked)]:
    print(label, {k: result[k] for k in ["input_rows", "accepted_rows", "rejected_rows", "promote_allowed"]})
print("Policy execution is not a Great Expectations run.")

trusted {'input_rows': 75, 'accepted_rows': 75, 'rejected_rows': 0, 'promote_allowed': True}
mixed {'input_rows': 82, 'accepted_rows': 75, 'rejected_rows': 7, 'promote_allowed': False}
rechecked {'input_rows': 75, 'accepted_rows': 75, 'rejected_rows': 0, 'promote_allowed': True}
Policy execution is not a Great Expectations run.


<table dir="ltr" width="100%">
<thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead>
<tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>5. Read all seven root reasons</h2></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>5. قراءة الأسباب الجذرية السبعة</h2></td></tr></tbody>
</table>



In [5]:
for rejected in mixed["quarantine"]:
    print(rejected["candidate_row"], rejected["trip_id"] or "<missing key>", ", ".join(rejected["reason_codes"]))
print(f"Rejected: {mixed['rejected_rows']} / {mixed['input_rows']} = {mixed['rejection_rate']:.4%}")
assert mixed["input_rows"] == mixed["accepted_rows"] + mixed["rejected_rows"]

76 <missing key> MISSING_TRIP_ID
77 SYN_BAD002 INVALID_FARE
78 SYN_BAD003 UNKNOWN_DRIVER
79 SYN_BAD004 INVALID_TIMESTAMP
80 SYN_BAD005 INVALID_DURATION
81 SYN_BAD006 INVALID_DISTANCE
82 SYN_BAD007 INVALID_CITY
Rejected: 7 / 82 = 8.5366%


<table dir="ltr" width="100%">
<thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead>
<tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>6. Use an explicit historical clock</h2></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>6. استخدام ساعة تاريخية صريحة</h2></td></tr></tbody>
</table>



In [6]:
print("As of:", reference["scenario_clock"])
print("Newest event:", reference["event_time_newest"])
print("Source event freshness:", reference["source_event_freshness"])
print("Delivery freshness:", reference["delivery_freshness"])
from masar.trust_reference import freshness
future_probe = freshness("2026-06-04T03:06:00Z", "2026-06-04T03:05:00Z", max_age_seconds=300)
assert future_probe["status"] == "FUTURE_TIMESTAMP"
print("A future timestamp is not automatically fresh:", future_probe)

As of: 2026-06-04T03:04:00Z
Newest event: 2026-06-03T17:45:00Z
Source event freshness: {'age_seconds': 33540, 'limit_seconds': 43200, 'status': 'PASS'}
Delivery freshness: {'age_seconds': 240, 'limit_seconds': 300, 'status': 'PASS'}
A future timestamp is not automatically fresh: {'age_seconds': -60, 'limit_seconds': 300, 'status': 'FUTURE_TIMESTAMP'}


<table dir="ltr" width="100%">
<thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead>
<tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>7. Interpret a distribution warning</h2></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>7. تفسير تنبيه التوزيع</h2></td></tr></tbody>
</table>



In [7]:
from copy import deepcopy
from masar.trust_reference import city_distribution
print("Accepted rows against Day 3:", city_distribution(rechecked["accepted"], trusted))
probe = deepcopy(trusted)
for row in probe:
    row["city"] = "Riyadh"
print("Transient example only, not a source edit:", city_distribution(probe, trusted))
assert rechecked["accepted"] == trusted

Accepted rows against Day 3: {'metric': 'categorical_total_variation_distance', 'value': 0.0, 'threshold': 0.1, 'status': 'PASS', 'current_counts': {'Riyadh': 25, 'Jeddah': 25, 'Dammam': 25}, 'baseline_counts': {'Riyadh': 25, 'Jeddah': 25, 'Dammam': 25}, 'meaning': 'Fixture-relative diagnostic; not statistical significance or real population drift.'}
Transient example only, not a source edit: {'metric': 'categorical_total_variation_distance', 'value': 0.6666666666666667, 'threshold': 0.1, 'status': 'WARN', 'current_counts': {'Riyadh': 75}, 'baseline_counts': {'Riyadh': 25, 'Jeddah': 25, 'Dammam': 25}, 'meaning': 'Fixture-relative diagnostic; not statistical significance or real population drift.'}


<table dir="ltr" width="100%">
<thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead>
<tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>8. Save the bounded evidence</h2></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>8. حفظ الأدلة المحدودة</h2></td></tr></tbody>
</table>



In [8]:
from masar.workspace import new_workspace, workspace_path, write_json
assert all(reference["checks"].values())
assert reference["engine_executed"] is reference["kafka_executed"] is reference["gx_executed"] is False
work = new_workspace(ROOT, "day04_reference")
evidence = workspace_path(work, "reports/day04_reference.json")
write_json(evidence, reference)
print("Reference checks:", len(reference["checks"]), "passed")
print("Preserved trusted amount:", reference["trusted_fare_sar"], "SAR")
print("Saved under a new outputs/day04_reference_* workspace.")
print("Native Kafka/Delta/GX, checkpoints and Data Docs still require actual execution.")

Reference checks: 13 passed
Preserved trusted amount: 1880.60 SAR
Saved under a new outputs/day04_reference_* workspace.
Native Kafka/Delta/GX, checkpoints and Data Docs still require actual execution.
